# Notebook 2: Exploratory Data Analysis

This notebook explores the processed dataset to understand distributions, relationships, and patterns before modelling. All plots are saved to `app/static/` so they can be embedded in the web interface.

**Input:** `data/processed_data.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Consistent style throughout
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11
})
BLUE = '#2c5f8a'
ORANGE = '#e07b39'

df = pd.read_csv('../data/processed_data.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 1. Target Variable Distribution

Energy consumption is extremely right-skewed — a handful of large economies dominate. The log-transformed version is approximately normal, which helps linear models and confirms the log transform used in feature engineering is appropriate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['primary_energy_consumption'], bins=60, color=BLUE, edgecolor='white', linewidth=0.4)
axes[0].set_title('Primary Energy Consumption (raw)')
axes[0].set_xlabel('TWh')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(df['primary_energy_consumption']), bins=60, color=ORANGE, edgecolor='white', linewidth=0.4)
axes[1].set_title('Primary Energy Consumption (log scale)')
axes[1].set_xlabel('log(TWh + 1)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../app/static/eda_target_dist.png', bbox_inches='tight')
plt.show()

skew_raw = stats.skew(df['primary_energy_consumption'])
skew_log = stats.skew(np.log1p(df['primary_energy_consumption']))
print(f"Skewness (raw): {skew_raw:.2f}")
print(f"Skewness (log): {skew_log:.2f}")

## 2. Top Consuming Countries Over Time

The top 6 consumers account for a disproportionate share of global energy. China's steep rise since the early 2000s is the most notable trend.

In [ ]:
top_countries = (
    df.groupby('country')['primary_energy_consumption']
    .mean()
    .sort_values(ascending=False)
    .head(6)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(11, 5))
colors = plt.cm.tab10.colors

for i, country in enumerate(top_countries):
    subset = df[df['country'] == country].sort_values('year')
    ax.plot(subset['year'], subset['primary_energy_consumption'],
            label=country, linewidth=2, color=colors[i])

ax.set_title('Primary Energy Consumption — Top 6 Countries (1990–2022)')
ax.set_xlabel('Year')
ax.set_ylabel('TWh')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('../app/static/eda_top_countries.png', bbox_inches='tight')
plt.show()

## 3. Correlation Heatmap

Looking at correlations between numeric features and the target to get a sense of which variables carry the most predictive signal.

In [ ]:
plot_cols = [
    'primary_energy_consumption', 'log_primary_energy_consumption',
    'population', 'log_population', 'gdp', 'log_gdp',
    'electricity_generation', 'coal_production', 'oil_production', 'gas_production',
    'renewables_share_elec', 'fossil_share_elec', 'year_norm'
]

corr = df[plot_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle only
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.4, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../app/static/eda_correlation.png', bbox_inches='tight')
plt.show()

## 4. Energy vs. GDP Relationship

The relationship between GDP and energy consumption is strongly positive but non-linear — richer countries consume more, but with diminishing returns at very high income levels. Colouring by year shows that for many countries, energy intensity has been improving over time.

In [ ]:
sample = df[(df['gdp'].notna()) & (df['year'].isin([1995, 2005, 2015, 2022]))]

fig, ax = plt.subplots(figsize=(10, 6))
year_colors = {1995: '#a8d5e2', 2005: '#5ba8c4', 2015: '#2c5f8a', 2022: '#e07b39'}

for yr, grp in sample.groupby('year'):
    ax.scatter(np.log1p(grp['gdp']), np.log1p(grp['primary_energy_consumption']),
               alpha=0.5, s=18, color=year_colors[yr], label=str(yr))

ax.set_xlabel('log(GDP + 1)')
ax.set_ylabel('log(Primary Energy + 1)')
ax.set_title('Energy Consumption vs. GDP (log scale, selected years)')
ax.legend(title='Year', frameon=False)
plt.tight_layout()
plt.savefig('../app/static/eda_gdp_energy.png', bbox_inches='tight')
plt.show()

## 5. Renewables Share Over Time

Global average renewables share has been growing, but there is large variation across countries. The box plot below shows the distribution per decade.

In [ ]:
df['decade'] = (df['year'] // 5) * 5  # group into 5-year bins

fig, ax = plt.subplots(figsize=(11, 5))

groups = [df[df['decade'] == d]['renewables_share_elec'].dropna() for d in sorted(df['decade'].unique())]
labels = [str(d) for d in sorted(df['decade'].unique())]

bp = ax.boxplot(groups, labels=labels, patch_artist=True,
                medianprops={'color': 'white', 'linewidth': 2},
                flierprops={'marker': '.', 'markersize': 3, 'alpha': 0.3})

for patch in bp['boxes']:
    patch.set_facecolor(BLUE)
    patch.set_alpha(0.7)

ax.set_xlabel('5-Year Period')
ax.set_ylabel('Renewables Share of Electricity (%)')
ax.set_title('Distribution of Renewables Share by Period')
plt.tight_layout()
plt.savefig('../app/static/eda_renewables_trend.png', bbox_inches='tight')
plt.show()

df.drop(columns='decade', inplace=True)

## 6. Feature–Target Scatter Grid

Quick scatter plots of the most important predictors vs. the log-transformed target.

In [ ]:
scatter_features = [
    ('log_population', 'log(Population)'),
    ('log_gdp', 'log(GDP)'),
    ('log_electricity_generation', 'log(Electricity Generation)'),
    ('renewables_share_elec', 'Renewables Share (%)'),
    ('fossil_share_elec', 'Fossil Share (%)'),
    ('year_norm', 'Year (normalised)')
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for ax, (feat, label) in zip(axes.flatten(), scatter_features):
    ax.scatter(df[feat], df['log_primary_energy_consumption'],
               alpha=0.15, s=8, color=BLUE)
    ax.set_xlabel(label)
    ax.set_ylabel('log(Energy)')
    # Add a trend line
    mask = df[[feat, 'log_primary_energy_consumption']].notna().all(axis=1)
    z = np.polyfit(df.loc[mask, feat], df.loc[mask, 'log_primary_energy_consumption'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(x_line, p(x_line), color=ORANGE, linewidth=1.5)

plt.suptitle('Feature vs. log(Energy Consumption)', y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig('../app/static/eda_scatter_grid.png', bbox_inches='tight')
plt.show()

## 7. Summary Statistics

In [ ]:
summary_cols = [
    'primary_energy_consumption', 'energy_per_capita',
    'population', 'gdp', 'electricity_generation',
    'renewables_share_elec', 'fossil_share_elec'
]
df[summary_cols].describe().round(2)

## 8. Multicollinearity Check (VIF)

Variance Inflation Factors are computed to identify highly collinear features. Features with VIF > 10 are flagged — this informs which features the linear baseline model may struggle with.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_features = [
    'log_population', 'log_gdp', 'log_electricity_generation',
    'coal_production', 'oil_production', 'gas_production',
    'renewables_share_elec', 'fossil_share_elec',
    'solar_share_elec', 'wind_share_elec', 'hydro_share_elec',
    'year_norm'
]

vif_df = df[vif_features].dropna()
vif_scores = pd.DataFrame({
    'Feature': vif_features,
    'VIF': [variance_inflation_factor(vif_df.values, i) for i in range(len(vif_features))]
}).sort_values('VIF', ascending=False)

print(vif_scores.to_string(index=False))
print(f"\nFeatures with VIF > 10: {(vif_scores['VIF'] > 10).sum()}")

## 9. Key Takeaways

From this EDA:

1. The target is **highly skewed** — log transformation is essential for linear models.
2. **Population and GDP** are the strongest individual predictors, but their relationship with energy is non-linear.
3. **Electricity generation** is nearly as informative as population, but avoids leakage since it captures the mix rather than total consumption.
4. **Renewables share** has been gradually increasing globally, but the variance across countries is large — this makes it a useful discriminating feature.
5. Some **multicollinearity** exists between the share columns (fossil + renewables ≈ 100%), but tree-based models handle this naturally.
6. The dataset has **no obvious data quality issues** after the imputation step — share columns are bounded, target is always positive.